# 📰 네이버 뉴스 크롤링 실전

앞서 배운 requests, BeautifulSoup, 정규표현식을 종합하여
네이버 뉴스 목록을 크롤링하는 실전 프로젝트입니다.

대상: [경향신문 뉴스 목록](https://news.naver.com/main/list.naver?mode=LPOD&mid=sec&oid=032)

```bash
pip install requests beautifulsoup4
```

| 항목 | 내용 |
|------|------|
| **사용 라이브러리** | requests, bs4 (BeautifulSoup), re |
| **대상 사이트** | 네이버 뉴스 경향신문 목록 |
| **추출 데이터** | 뉴스 제목, 링크, 등록시간 |

---
## 1. 웹 페이지 요청하기

네이버 뉴스 페이지에 HTTP GET 요청을 보냅니다.
뉴스 사이트는 봇 접근을 차단할 수 있으므로 **User-Agent 헤더**를 설정합니다.

In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import json
from datetime import datetime

# 네이버 뉴스 경향신문 목록 URL
url = 'https://news.naver.com/main/list.naver?mode=LPOD&mid=sec&oid=032'

# User-Agent 헤더 설정 (브라우저처럼 보이게)
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'ko-KR,ko;q=0.9,en-US;q=0.8',
}

response = requests.get(url, headers=headers, timeout=10)
response.raise_for_status()
print(f'상태 코드: {response.status_code}')
print(f'콘텐츠 타입: {response.headers.get("Content-Type")}')
print(f'인코딩: {response.encoding}')
print(f'HTML 길이: {len(response.text)}자')

---
## 2. HTML 구조 파악하기

BeautifulSoup으로 HTML을 파싱하고, 뉴스 목록의 구조를 파악합니다.
네이버 뉴스 목록 페이지의 주요 구조:

```
<div class="list_body">
  <ul> <!-- 뉴스 목록 -->
    <li> <!-- 개별 뉴스 항목 -->
      <a href="...">뉴스 제목</a>
    </li>
  </ul>
</div>
```

In [ ]:
# BeautifulSoup으로 파싱
soup = BeautifulSoup(response.text, 'html.parser')

# 페이지 제목 확인
title = soup.find('title')
print(f'페이지 제목: {title.get_text(strip=True) if title else "없음"}')

# 뉴스 목록 컨테이너 찾기
# 네이버 뉴스는 class="list_body" 영역에 뉴스 목록이 있습니다
list_body = soup.find('div', class_='list_body')
if list_body:
    print(f'list_body 발견! 하위 요소 개수: {len(list_body.find_all("li"))}')
else:
    print('list_body 없음 — 다른 구조 확인 필요')
    # 대체: 페이지 내 링크 구조 확인
    all_links = soup.find_all('a', href=True)
    news_links = [a for a in all_links if 'article' in a.get('href', '')]
    print(f'기사 링크 수: {len(news_links)}')
    for a in news_links[:5]:
        print(f'  {a.get_text(strip=True)[:50]}')
        print(f'    href: {a["href"][:80]}')

---
## 3. 뉴스 목록 추출하기

뉴스 제목과 링크를 체계적으로 추출합니다.
중복 제거와 데이터 정제 과정을 포함합니다.

In [ ]:
# 뉴스 목록 추출 함수
def extract_news_list(soup):
    """BeautifulSoup 객체에서 뉴스 목록을 추출합니다."""
    news_list = []
    seen_titles = set()  # 중복 제거용

    # 방법 1: list_body 내의 링크 찾기
    list_body = soup.find('div', class_='list_body')
    if list_body:
        links = list_body.find_all('a', href=True)
    else:
        # 방법 2: article이 포함된 모든 링크
        links = soup.find_all('a', href=re.compile(r'article'))

    for link in links:
        title = link.get_text(strip=True)
        href = link['href']

        # 빈 제목이나 너무 짧은 제목 건너뛰기
        if not title or len(title) < 5:
            continue
        # 중복 제거
        if title in seen_titles:
            continue
        seen_titles.add(title)

        # 상대 경로를 절대 경로로 변환
        if href.startswith('/'):
            href = 'https://news.naver.com' + href

        news_list.append({
            'title': title,
            'url': href
        })

    return news_list

# 실행
news = extract_news_list(soup)
print(f'추출된 뉴스 수: {len(news)}')
print('\n--- 최신 뉴스 목록 ---')
for i, item in enumerate(news[:10]):
    print(f'{i+1}. {item["title"][:60]}')
    print(f'   {item["url"][:80]}')

---
## 4. 정규표현식으로 데이터 정제

추출한 데이터에서 불필요한 문자를 제거하고,
정규표현식으로 추가 정보를 파악합니다.

In [ ]:
# 정규표현식으로 데이터 정제
print('--- 정제 전 샘플 ---')
for item in news[:3]:
    print(f'  제목: "{item["title"]}"')

# HTML 엔티티 및 특수문자 제거
def clean_title(title):
    # &nbsp;, &amp; 등 HTML 엔티티 제거
    title = re.sub(r'&\w+;', ' ', title)
    # 연속 공백을 하나로
    title = re.sub(r'\s+', ' ', title).strip()
    # 대괄호 내용 제거 (예: [속보], [단독] 등은 유지)
    # title = re.sub(r'\[[^\]]+\]', '', title)
    return title

for item in news:
    item['title'] = clean_title(item['title'])

print('\n--- 정제 후 ---')
for i, item in enumerate(news[:5]):
    print(f'{i+1}. {item["title"]}')

# URL에서 기사 ID 추출
for item in news[:3]:
    m = re.search(r'(\d+)$', item['url'])
    if m:
        print(f'  기사 ID: {m.group(1)}')

---
## 5. 결과를 정리하여 출력

추출한 뉴스 데이터를 깔끔하게 정리하여 출력하고,
JSON 파일로 저장합니다.

In [ ]:
# 최종 결과 정리
print('=' * 60)
print(f'네이버 뉴스 경향신문 목록 ({len(news)}건)')
print('=' * 60)

for i, item in enumerate(news, 1):
    print(f'\n[{i}] {item["title"]}')
    print(f'    URL: {item["url"]}')

# JSON 파일로 저장
output = {
    'source': '네이버 뉴스 경향신문',
    'url': url,
    'count': len(news),
    'news': news
}

output_file = 'naver_news_kyunghyang.json'
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f'\n저장 완료: {output_file}')
print(f'총 {len(news)}건의 뉴스가 저장되었습니다.')

---
## 6. 예외 처리와 재시도 로직

실제 크롤링에서는 네트워크 오류, 서버 응답 지연, HTML 구조 변경 등
다양한 문제가 발생할 수 있습니다. 안정적인 크롤링을 위한 예외 처리 패턴입니다.

In [ ]:
def fetch_news_with_retry(url, headers, max_retries=3, delay=2):
    """재시도 로직이 포함된 HTTP 요청 함수"""
    import time
    from requests.exceptions import Timeout, ConnectionError, HTTPError

    for attempt in range(1, max_retries + 1):
        try:
            print(f'  시도 {attempt}/{max_retries}...')
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status()
            return response

        except Timeout:
            print(f'  ⏰ 타임아웃 (5초)')
        except ConnectionError:
            print(f'  🔌 연결 실패')
        except HTTPError as e:
            print(f'  ❌ HTTP 에러: {e.response.status_code}')
            if e.response.status_code == 404:
                return None  # 404는 재시도 불필요
        except Exception as e:
            print(f'  ⚠️ 알 수 없는 오류: {e}')

        if attempt < max_retries:
            print(f'  {delay}초 후 재시도...')
            time.sleep(delay)

    print('  ❌ 최대 재시도 횟수 초과')
    return None

# 실행
print('재시도 로직 테스트:')
response = fetch_news_with_retry(url, headers)
if response:
    print(f'✅ 성공! 상태 코드: {response.status_code}')
else:
    print('❌ 요청 실패')

---
## 7. 종합: 전체 크롤링 파이프라인

앞의 모든 기술을 합친 완전한 크롤링 스크립트입니다.
요청 → 파싱 → 추출 → 정제 → 저장의 파이프라인으로 구성됩니다.

In [ ]:
def crawl_naver_news(url, headers=None):
    """네이버 뉴스 크롤링 전체 파이프라인"""
    if headers is None:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                          'AppleWebKit/537.36 (KHTML, like Gecko) '
                          'Chrome/120.0.0.0 Safari/537.36',
            'Accept-Language': 'ko-KR,ko;q=0.9',
        }

    # 1단계: 요청
    print('1️⃣ 페이지 요청...')
    response = fetch_news_with_retry(url, headers)
    if response is None:
        return []

    # 2단계: 파싱
    print('2️⃣ HTML 파싱...')
    soup = BeautifulSoup(response.text, 'html.parser')

    # 3단계: 추출
    print('3️⃣ 뉴스 목록 추출...')
    news = extract_news_list(soup)

    # 4단계: 정제
    print('4️⃣ 데이터 정제...')
    for item in news:
        item['title'] = clean_title(item['title'])

    # 5단계: 저장
    print(f'5️⃣ 결과: {len(news)}건 추출 완료')
    return news

# 실행
results = crawl_naver_news(url)
print(f'\n총 {len(results)}건의 뉴스를 추출했습니다.')
for i, item in enumerate(results[:5], 1):
    print(f'  {i}. {item["title"][:50]}')

---
## 🎯 연습 문제

배운 내용을 종합하여 실습해 보세요.

1. `https://httpbin.org/links/3/0` 페이지에서 모든 링크를 추출하세요.

2. `https://httpbin.org/html` 페이지에서 모든 `<p>` 태그의 텍스트를 추출하고, 각 문단의 글자 수를 출력하세요.

3. 정규표현식을 사용해 `https://httpbin.org/html` 페이지에서 모든 이메일 주소 패턴을 찾으세요 (이메일이 없다면 어떤 패턴이 있는지 확인).

4. 임의의 HTML 문자열에서 `<table>`의 모든 행 데이터를 리스트로 추출하는 함수를 작성하세요.

5. (도전) 네이버 뉴스 페이지에서 뉴스 제목을 추출할 때, 타이틀이 20자 이상인 뉴스만 `[20자이상]` 태그를 붙여 출력하는 코드를 작성하세요.

In [ ]:
# 연습 문제 — 직접 코드를 작성해 보세요
import requests
from bs4 import BeautifulSoup
import re

# 1. httpbin 링크 추출

# 2. httpbin/html 페이지 문단 추출

# 3. 이메일/패턴 검색

# 4. 테이블 데이터 추출 함수

# 5. (도전) 20자 이상 뉴스 타이틀

---
## 🎯 정리

이 튜토리얼에서 배운 크롤링 파이프라인:

```
URL 요청 → HTML 파싱 → 데이터 추출 → 정제 → 저장
requests    BeautifulSoup  정규표현식  JSON
```

| 단계 | 라이브러리 | 핵심 함수 |
|------|-----------|----------|
| 1. 요청 | requests | `requests.get(url, headers=)` |
| 2. 파싱 | BeautifulSoup | `BeautifulSoup(html, 'html.parser')` |
| 3. 검색 | BeautifulSoup | `find()`, `find_all()`, `select()` |
| 4. 정제 | re | `re.sub()`, `re.search()` |
| 5. 저장 | json | `json.dump()` |

⚠️ **크롤링 주의사항**:
- 웹사이트의 `robots.txt`를 확인하세요
- 과도한 요청은 서버에 부하를 줍니다 (1초 이상 간격 권장)
- 저작권과 이용약관을 준수하세요
- API가 제공되면 API를 우선 사용하세요